In [1]:
import pandas as pd
import os

## Import the data

In [2]:
df_raw = pd.read_csv("linkedin_job_cleaned_merged_small_0211.csv")

df_raw.head()

,Unnamed: 0,job_link,got_summary,got_ner,is_being_worked,job_title,company,job_location,first_seen,search_city,...,search_position,job_level,job_type,city,state,country,city_state,latitude,longitude,job_skills
0,0,https://www.linkedin.com/jobs/view/warehouse-a...,t,t,f,Warehouse Associate/ driver/ Inventory Control,Infrahire,"['Groveport', 'OH', 'United States']",2024-01-12,Columbus,...,Chauffeur,Associate,Onsite,Groveport,OH,United States,"Groveport, OH",39.857337,-82.891442,"Warehouse Management, Inventory Control, Mater..."
1,1,https://www.linkedin.com/jobs/view/sales-super...,t,t,f,Sales Supervisor - Spencer's,Spencer's,"['Jenks', 'OK', 'United States']",2024-01-14,Sun Valley,...,Superintendent Sales,Mid senior,Onsite,Jenks,OK,United States,"Jenks, OK",35.998219,-95.973404,"Sales, Customer Service, Merchandising, Invent..."
2,2,https://www.linkedin.com/jobs/view/registered-...,t,t,f,"Registered Nurse (Women, Infants, & Children) ...",Health eCareers,"['Springfield', 'MO', 'United States']",2024-01-12,Spokane,...,Christian Science Nurse,Mid senior,Onsite,Springfield,MO,United States,"Springfield, MO",37.194157,-93.292642,"Nursing, Registered Nurse license, Client care..."
3,3,https://www.linkedin.com/jobs/view/customer-se...,t,t,f,CUSTOMER SERVICE REPRESENTATIVE,Family Dollar,"['Tyrone', 'PA', 'United States']",2024-01-12,State College,...,Car Inspector,Mid senior,Onsite,Tyrone,PA,United States,"Tyrone, PA",40.676451,-78.246044,"Customer service, POS system, Stocking shelves..."
4,4,https://www.linkedin.com/jobs/view/customer-se...,t,t,f,Customer Service Rep,Lincare,"['Daytona Beach', 'FL', 'United States']",2024-01-12,Daytona Beach,...,Car Inspector,Mid senior,Onsite,Daytona Beach,FL,United States,"Daytona Beach, FL",29.190710,-81.097083,"Patient care, Chronic care, Medical terminolog..."


# Basic procedure

In [3]:
df = df_raw

df.drop(columns=["Unnamed: 0", "job_link"], inplace=True, errors="ignore")
df = df[df["got_summary"] == "t"]
df.drop(columns=["got_summary"], inplace=True)
df.drop(columns=["job_location"], inplace=True)
df.drop(columns=["is_being_worked"], inplace=True)
df["first_seen"] = pd.to_datetime(df["first_seen"])
df.drop(columns=["got_ner"], inplace=True)
df.drop(columns=["search_city"], inplace=True)

In [4]:
df = df.drop(columns=['first_seen', 'search_country', 'city_state', 'country'])

Processing on Skills

In [5]:
df["job_skills_list"] = df["job_skills"].apply(
    lambda x: [s.strip() for s in x.split(",")] if isinstance(x, str) else []
)

In [6]:
df.head(10)

,job_title,company,search_position,job_level,job_type,city,state,latitude,longitude,job_skills,job_skills_list
0,Warehouse Associate/ driver/ Inventory Control,Infrahire,Chauffeur,Associate,Onsite,Groveport,OH,39.857337,-82.891442,"Warehouse Management, Inventory Control, Mater...","[Warehouse Management, Inventory Control, Mate..."
1,Sales Supervisor - Spencer's,Spencer's,Superintendent Sales,Mid senior,Onsite,Jenks,OK,35.998219,-95.973404,"Sales, Customer Service, Merchandising, Invent...","[Sales, Customer Service, Merchandising, Inven..."
2,"Registered Nurse (Women, Infants, & Children) ...",Health eCareers,Christian Science Nurse,Mid senior,Onsite,Springfield,MO,37.194157,-93.292642,"Nursing, Registered Nurse license, Client care...","[Nursing, Registered Nurse license, Client car..."
3,CUSTOMER SERVICE REPRESENTATIVE,Family Dollar,Car Inspector,Mid senior,Onsite,Tyrone,PA,40.676451,-78.246044,"Customer service, POS system, Stocking shelves...","[Customer service, POS system, Stocking shelve..."
4,Customer Service Rep,Lincare,Car Inspector,Mid senior,Onsite,Daytona Beach,FL,29.190710,-81.097083,"Patient care, Chronic care, Medical terminolog...","[Patient care, Chronic care, Medical terminolo..."
5,Corporate Receptionist for Northwest Center Se...,Northwest Center,Check Cashier,Mid senior,Onsite,Columbia,MD,39.201146,-76.859050,"Customer service, Call center, Administrative ...","[Customer service, Call center, Administrative..."
6,Assistant Manager (Full Time),Tillys,Manager Customer Service,Mid senior,Onsite,Woodbridge,NJ,40.552857,-74.286939,"Store Management, Customer Service, Hiring, Tr...","[Store Management, Customer Service, Hiring, T..."
7,Associate Project Manager,LAKE SUPERIOR CONSULTING,Charter,Mid senior,Onsite,Canonsburg,PA,40.264246,-80.186776,"Project Management, Project Coordination, CAPM...","[Project Management, Project Coordination, CAP..."
8,Member Services Advisor,State of Michigan,Car Inspector,Mid senior,Onsite,Lansing,MI,42.714341,-84.560889,"Legislative affairs, Public relations, Writing...","[Legislative affairs, Public relations, Writin..."
9,Assistant Buyer,Detroit Job Corps Center,Buyer Assistant,Mid senior,Onsite,Detroit,MI,42.383037,-83.102237,"Purchasing, Microsoft Office (Word PowerPoint ...","[Purchasing, Microsoft Office (Word PowerPoint..."


# Industry Categorization

In [7]:
# categorizing the job postings based on search_name from Linkedin

def categorize_industry_from_skills(skills):
    if not skills:
        return "Other Services (except Public Administration)"
    
    if isinstance(skills, list):
        s = " ".join(skills).lower()
    else:
        s = str(skills).lower()

    # 92 Public Administration
    if (
        "government" in s or "public administration" in s or
        "city of" in s or "county" in s or "state of" in s or
        "federal" in s or "agency" in s or "department of" in s or
        "municipal" in s or "public sector" in s
    ):
        return "Public Administration"

    # 11 Agriculture, Forestry, Fishing and Hunting
    if (
        "agricultur" in s or "farm" in s or "ranch" in s or
        "forestry" in s or "logging" in s or
        "fishing" in s or "fishery" in s or "hunting" in s
    ):
        return "Agriculture, Forestry, Fishing and Hunting"

    # 21 Mining, Quarrying, and Oil and Gas Extraction
    if (
        "mining" in s or "quarry" in s or "ore" in s or
        "oil" in s or "gas" in s or "petroleum" in s or
        "drilling" in s or "rig" in s or "pipeline extraction" in s
    ):
        return "Mining, Quarrying, and Oil and Gas Extraction"

    # 22 Utilities
    if (
        "utility" in s or "electric" in s or "power" in s or
        "water" in s or "sewage" in s or "wastewater" in s or
        "gas utility" in s or "grid" in s
    ):
        return "Utilities"

    # 23 Construction
    if (
        "construction" in s or "contractor" in s or "builder" in s or
        "plumbing" in s or "hvac" in s or "roof" in s or
        "carpenter" in s or "electrician" in s or "remodel" in s
    ):
        return "Construction"

    # 31-33 Manufacturing
    if (
        "manufactur" in s or "factory" in s or "plant" in s or
        "assembly" in s or "fabricat" in s or "production" in s or
        "packaging" in s
    ):
        return "Manufacturing"

    # 42 Wholesale Trade
    if (
        "wholesale" in s or "distributor" in s or
        "b2b sales" in s or "wholesaler" in s
    ):
        return "Wholesale Trade"

    # 44-45 Retail Trade
    if (
        "retail" in s or "store" in s or "shop" in s or
        "cashier" in s or "sales associate" in s or
        "merchandis" in s or "e-commerce" in s or "ecommerce" in s
    ):
        return "Retail Trade"

    # 48-49 Transportation and Warehousing
    if (
        "transport" in s or "logistics" in s or "warehouse" in s or
        "delivery" in s or "driver" in s or "trucking" in s or
        "shipping" in s or "freight" in s or "courier" in s
    ):
        return "Transportation and Warehousing"

    # 51 Information
    if (
        "software" in s or "it " in s or " information " in s or
        "data" in s or "cloud" in s or "saas" in s or
        "telecom" in s or "internet" in s or "media" in s or
        "publishing" in s or "broadcast" in s or "web" in s
    ):
        return "Information"

    # 52 Finance and Insurance
    if (
        "finance" in s or "financial" in s or "bank" in s or
        "investment" in s or "trading" in s or "trader" in s or
        "accounting" in s or "audit" in s or "tax" in s or
        "insurance" in s or "actuar" in s or "credit" in s or
        "fp&a" in s or "treasury" in s
    ):
        return "Finance and Insurance"

    # 53 Real Estate and Rental and Leasing
    if (
        "real estate" in s or "property" in s or "leasing" in s or
        "rental" in s or "mortgage" in s or "realtor" in s
    ):
        return "Real Estate and Rental and Leasing"

    # 54 Professional, Scientific, and Technical Services
    if (
        "consult" in s or "consulting" in s or
        "legal" in s or "law" in s or "attorney" in s or
        "architect" in s or "engineering service" in s or
        "advertis" in s or "design agency" in s or
        "research" in s or "laboratory" in s
    ):
        return "Professional, Scientific, and Technical Services"

    # 55 Management of Companies and Enterprises
    if (
        "holding company" in s or "holdings" in s or
        "headquarters" in s or "hq" in s or
        "management of companies" in s
    ):
        return "Management of Companies and Enterprises"

    # 56 Administrative and Support and Waste Management and Remediation Services
    if (
        "staffing" in s or "recruit" in s or "temp agency" in s or
        "facility services" in s or "janitor" in s or "cleaning" in s or
        "security guard" in s or "waste management" in s or
        "remediation" in s or "call center" in s
    ):
        return "Administrative and Support and Waste Management and Remediation Services"

    # 61 Educational Services
    if (
        "school" in s or "university" in s or "college" in s or
        "education" in s or "teacher" in s or "tutor" in s or
        "instructor" in s or "professor" in s
    ):
        return "Educational Services"

    # 62 Health Care and Social Assistance
    if (
        "hospital" in s or "clinic" in s or "medical" in s or
        "health" in s or "nurse" in s or "physician" in s or
        "pharmacy" in s or "dental" in s or "therapy" in s or
        "caregiver" in s or "social assistance" in s
    ):
        return "Health Care and Social Assistance"

    # 71 Arts, Entertainment, and Recreation
    if (
        "museum" in s or "arts" in s or "theater" in s or
        "music" in s or "entertainment" in s or "sports" in s or
        "recreation" in s or "casino" in s or "gym" in s
    ):
        return "Arts, Entertainment, and Recreation"

    # 72 Accommodation and Food Services
    if (
        "hotel" in s or "hospitality" in s or "resort" in s or
        "restaurant" in s or "food" in s or "cafe" in s or
        "bar" in s or "kitchen" in s or "catering" in s
    ):
        return "Accommodation and Food Services"

    # 81 Other Services
    return "Other Services (except Public Administration)"

In [8]:
df["industry_group"] = df["job_skills_list"].apply(categorize_industry_from_skills)

In [9]:
df.head(20)

,job_title,company,search_position,job_level,job_type,city,state,latitude,longitude,job_skills,job_skills_list,industry_group
0,Warehouse Associate/ driver/ Inventory Control,Infrahire,Chauffeur,Associate,Onsite,Groveport,OH,39.857337,-82.891442,"Warehouse Management, Inventory Control, Mater...","[Warehouse Management, Inventory Control, Mate...",Transportation and Warehousing
1,Sales Supervisor - Spencer's,Spencer's,Superintendent Sales,Mid senior,Onsite,Jenks,OK,35.998219,-95.973404,"Sales, Customer Service, Merchandising, Invent...","[Sales, Customer Service, Merchandising, Inven...","Mining, Quarrying, and Oil and Gas Extraction"
2,"Registered Nurse (Women, Infants, & Children) ...",Health eCareers,Christian Science Nurse,Mid senior,Onsite,Springfield,MO,37.194157,-93.292642,"Nursing, Registered Nurse license, Client care...","[Nursing, Registered Nurse license, Client car...",Health Care and Social Assistance
3,CUSTOMER SERVICE REPRESENTATIVE,Family Dollar,Car Inspector,Mid senior,Onsite,Tyrone,PA,40.676451,-78.246044,"Customer service, POS system, Stocking shelves...","[Customer service, POS system, Stocking shelve...",Retail Trade
4,Customer Service Rep,Lincare,Car Inspector,Mid senior,Onsite,Daytona Beach,FL,29.190710,-81.097083,"Patient care, Chronic care, Medical terminolog...","[Patient care, Chronic care, Medical terminolo...",Information
5,Corporate Receptionist for Northwest Center Se...,Northwest Center,Check Cashier,Mid senior,Onsite,Columbia,MD,39.201146,-76.859050,"Customer service, Call center, Administrative ...","[Customer service, Call center, Administrative...",Finance and Insurance
6,Assistant Manager (Full Time),Tillys,Manager Customer Service,Mid senior,Onsite,Woodbridge,NJ,40.552857,-74.286939,"Store Management, Customer Service, Hiring, Tr...","[Store Management, Customer Service, Hiring, T...","Mining, Quarrying, and Oil and Gas Extraction"
7,Associate Project Manager,LAKE SUPERIOR CONSULTING,Charter,Mid senior,Onsite,Canonsburg,PA,40.264246,-80.186776,"Project Management, Project Coordination, CAPM...","[Project Management, Project Coordination, CAP...",Other Services (except Public Administration)
8,Member Services Advisor,State of Michigan,Car Inspector,Mid senior,Onsite,Lansing,MI,42.714341,-84.560889,"Legislative affairs, Public relations, Writing...","[Legislative affairs, Public relations, Writin...",Information
9,Assistant Buyer,Detroit Job Corps Center,Buyer Assistant,Mid senior,Onsite,Detroit,MI,42.383037,-83.102237,"Purchasing, Microsoft Office (Word PowerPoint ...","[Purchasing, Microsoft Office (Word PowerPoint...",Utilities


In [10]:
from collections import Counter

# For each company, find the industry that they have the most job postings on Linkedin (except for "Other"), if no industry found, return other

def mode_exclude_other(arr):
    arr = [x for x in arr if x != "Other Services (except Public Administration)"]
    
    if len(arr) == 0:
        return "Other Services (except Public Administration)"
    
    counts = Counter(arr)
    return counts.most_common(1)[0][0]

In [11]:
company_to_major_industry = (
    df.groupby("company")["industry_group"]
    .apply(mode_exclude_other)
    .to_dict()
)

In [12]:
df["company_major_industry"] = (
    df["company"]
    .map(company_to_major_industry)
    .fillna("Other Services (except Public Administration)")
)

In [13]:
# drop job_skills
df_major = df.copy()
df_major = df.drop(columns=["job_skills"], errors="ignore")
df_major = df.drop(columns=["industry_group"], errors="ignore")

In [14]:
df.head(10)

,job_title,company,search_position,job_level,job_type,city,state,latitude,longitude,job_skills,job_skills_list,industry_group,company_major_industry
0,Warehouse Associate/ driver/ Inventory Control,Infrahire,Chauffeur,Associate,Onsite,Groveport,OH,39.857337,-82.891442,"Warehouse Management, Inventory Control, Mater...","[Warehouse Management, Inventory Control, Mate...",Transportation and Warehousing,Construction
1,Sales Supervisor - Spencer's,Spencer's,Superintendent Sales,Mid senior,Onsite,Jenks,OK,35.998219,-95.973404,"Sales, Customer Service, Merchandising, Invent...","[Sales, Customer Service, Merchandising, Inven...","Mining, Quarrying, and Oil and Gas Extraction",Retail Trade
2,"Registered Nurse (Women, Infants, & Children) ...",Health eCareers,Christian Science Nurse,Mid senior,Onsite,Springfield,MO,37.194157,-93.292642,"Nursing, Registered Nurse license, Client care...","[Nursing, Registered Nurse license, Client car...",Health Care and Social Assistance,Health Care and Social Assistance
3,CUSTOMER SERVICE REPRESENTATIVE,Family Dollar,Car Inspector,Mid senior,Onsite,Tyrone,PA,40.676451,-78.246044,"Customer service, POS system, Stocking shelves...","[Customer service, POS system, Stocking shelve...",Retail Trade,Retail Trade
4,Customer Service Rep,Lincare,Car Inspector,Mid senior,Onsite,Daytona Beach,FL,29.190710,-81.097083,"Patient care, Chronic care, Medical terminolog...","[Patient care, Chronic care, Medical terminolo...",Information,Information
5,Corporate Receptionist for Northwest Center Se...,Northwest Center,Check Cashier,Mid senior,Onsite,Columbia,MD,39.201146,-76.859050,"Customer service, Call center, Administrative ...","[Customer service, Call center, Administrative...",Finance and Insurance,Finance and Insurance
6,Assistant Manager (Full Time),Tillys,Manager Customer Service,Mid senior,Onsite,Woodbridge,NJ,40.552857,-74.286939,"Store Management, Customer Service, Hiring, Tr...","[Store Management, Customer Service, Hiring, T...","Mining, Quarrying, and Oil and Gas Extraction",Retail Trade
7,Associate Project Manager,LAKE SUPERIOR CONSULTING,Charter,Mid senior,Onsite,Canonsburg,PA,40.264246,-80.186776,"Project Management, Project Coordination, CAPM...","[Project Management, Project Coordination, CAP...",Other Services (except Public Administration),"Mining, Quarrying, and Oil and Gas Extraction"
8,Member Services Advisor,State of Michigan,Car Inspector,Mid senior,Onsite,Lansing,MI,42.714341,-84.560889,"Legislative affairs, Public relations, Writing...","[Legislative affairs, Public relations, Writin...",Information,Health Care and Social Assistance
9,Assistant Buyer,Detroit Job Corps Center,Buyer Assistant,Mid senior,Onsite,Detroit,MI,42.383037,-83.102237,"Purchasing, Microsoft Office (Word PowerPoint ...","[Purchasing, Microsoft Office (Word PowerPoint...",Utilities,Utilities


In [15]:
df

,job_title,company,search_position,job_level,job_type,city,state,latitude,longitude,job_skills,job_skills_list,industry_group,company_major_industry
0,Warehouse Associate/ driver/ Inventory Control,Infrahire,Chauffeur,Associate,Onsite,Groveport,OH,39.857337,-82.891442,"Warehouse Management, Inventory Control, Mater...","[Warehouse Management, Inventory Control, Mate...",Transportation and Warehousing,Construction
1,Sales Supervisor - Spencer's,Spencer's,Superintendent Sales,Mid senior,Onsite,Jenks,OK,35.998219,-95.973404,"Sales, Customer Service, Merchandising, Invent...","[Sales, Customer Service, Merchandising, Inven...","Mining, Quarrying, and Oil and Gas Extraction",Retail Trade
2,"Registered Nurse (Women, Infants, & Children) ...",Health eCareers,Christian Science Nurse,Mid senior,Onsite,Springfield,MO,37.194157,-93.292642,"Nursing, Registered Nurse license, Client care...","[Nursing, Registered Nurse license, Client car...",Health Care and Social Assistance,Health Care and Social Assistance
3,CUSTOMER SERVICE REPRESENTATIVE,Family Dollar,Car Inspector,Mid senior,Onsite,Tyrone,PA,40.676451,-78.246044,"Customer service, POS system, Stocking shelves...","[Customer service, POS system, Stocking shelve...",Retail Trade,Retail Trade
4,Customer Service Rep,Lincare,Car Inspector,Mid senior,Onsite,Daytona Beach,FL,29.190710,-81.097083,"Patient care, Chronic care, Medical terminolog...","[Patient care, Chronic care, Medical terminolo...",Information,Information
...,...,...,...,...,...,...,...,...,...,...,...,...,...
934275,HOUSEKEEPER (PART TIME),Crothall Healthcare,Cleaner,Mid senior,Onsite,Kaufman,TX,32.577249,-96.313715,"Housekeeping, Cleaning, Vacuuming, Polishing, ...","[Housekeeping, Cleaning, Vacuuming, Polishing,...",Administrative and Support and Waste Managemen...,Administrative and Support and Waste Managemen...
934276,Assistant General Manager,McDonald's,Clerk General,Mid senior,Onsite,Fort Wayne,IN,41.088193,-85.142791,"Leadership, Communication, Food Safety, Kitche...","[Leadership, Communication, Food Safety, Kitch...",Accommodation and Food Services,Manufacturing
934277,Permanent | Physician Family Practice,CompHealth,Family Practitioner,Mid senior,Onsite,Lewiston,ME,44.089513,-70.172095,"Flexible scheduling, Benefits, Salary negotiat...","[Flexible scheduling, Benefits, Salary negotia...",Other Services (except Public Administration),Health Care and Social Assistance
934278,Family Nurse Practitioner,Monte Nido & Affiliates,Family Practitioner,Mid senior,Onsite,Dedham,MA,42.246872,-71.179462,"Family Nurse Practitioner (FNP), ANCC Board Ce...","[Family Nurse Practitioner (FNP), ANCC Board C...",Educational Services,Health Care and Social Assistance


In [20]:
# Restrict to top 8 industries by job count (smaller dataset)
TOP_N_INDUSTRIES = 4
top_industries = df["company_major_industry"].value_counts().head(TOP_N_INDUSTRIES)
top_4_industry_names = top_industries.index.tolist()

In [21]:
top_4_industry_names

['Information',
 'Health Care and Social Assistance',
 'Retail Trade',
 'Utilities']

In [22]:
df = df[df["company_major_industry"].isin(top_4_industry_names)].copy()
print(f"Top {TOP_N_INDUSTRIES} industries: {top_4_industry_names}")
print(f"Filtered df shape: {df.shape}")

Top 4 industries: ['Information', 'Health Care and Social Assistance', 'Retail Trade', 'Utilities']
Filtered df shape: (523314, 13)


# Processing for Exporting

In [23]:
df_export = df

In [24]:
state_counts = df_export["state"].value_counts().reset_index()
state_counts.columns = ["state", "count"]

state_counts.head(30)

,state,count
0,CA,64162
1,TX,41942
2,FL,36580
3,PA,22905
4,NY,22897
5,OH,22360
6,VA,19946
7,NC,19560
8,MA,16498
9,IL,15743


Split the huge dataframe by state into several datasets with approriate size 

In [25]:
def export_state_data(df, state_code, drop_cols=None):
    
    base_path = "/Users/gloria/Cse-412/Data/Processed Data/0308_update"
    
    df_state = df[df["state"] == state_code].copy().reset_index(drop=True)
    
    if drop_cols is not None:
        df_state.drop(columns=drop_cols, inplace=True, errors="ignore")

    output_path = f"{base_path}/linkedin_jobs_{state_code}.csv"
    
    df_state.to_csv(output_path, index=False)
    
    print(f"{state_code} exported to:", output_path)
    print("Rows:", df_state.shape[0])
    
    return df_state

In [26]:
states = ["CA", "TX", "FL"]

for state in states:
    export_state_data(df_export, state)


CA exported to: /Users/gloria/Cse-412/Data/Processed Data/0308_update/linkedin_jobs_CA.csv
Rows: 64162
TX exported to: /Users/gloria/Cse-412/Data/Processed Data/0308_update/linkedin_jobs_TX.csv
Rows: 41942
FL exported to: /Users/gloria/Cse-412/Data/Processed Data/0308_update/linkedin_jobs_FL.csv
Rows: 36580


In [27]:
base_path = "/Users/gloria/Cse-412/Data/Processed Data/0308_update"
os.makedirs(base_path, exist_ok=True)

def export_states_in_groups(df, states, group_size=3):
    
    for i in range(0, len(states), group_size):
        
        group = states[i:i+group_size]
        
        df_group = df[df["state"].isin(group)].copy().reset_index(drop=True)
        
        filename = f"linkedin_jobs_{'_'.join(group)}.csv"
        output_path = f"{base_path}/{filename}"
        
        df_group.to_csv(output_path, index=False)
        
        print(f"Exported: {filename} | Rows: {df_group.shape[0]}")


In [28]:
states = [
    "PA","OH","NY",
    "VA","NC","MA",
    "IL","GA","MI",
    "WA","AZ","NJ",
    "CO","MD","IN",
    "WI","SC","TN",
    "MN","AL",
    "MO","IA","LA",
    "CT","NV","OK"
]


export_states_in_groups(df_export, states)

Exported: linkedin_jobs_PA_OH_NY.csv | Rows: 68162
Exported: linkedin_jobs_VA_NC_MA.csv | Rows: 56004
Exported: linkedin_jobs_IL_GA_MI.csv | Rows: 44959
Exported: linkedin_jobs_WA_AZ_NJ.csv | Rows: 38198
Exported: linkedin_jobs_CO_MD_IN.csv | Rows: 33139
Exported: linkedin_jobs_WI_SC_TN.csv | Rows: 26361
Exported: linkedin_jobs_MN_AL_MO.csv | Rows: 23640
Exported: linkedin_jobs_IA_LA_CT.csv | Rows: 20669
Exported: linkedin_jobs_NV_OK.csv | Rows: 9790


# Group By for Q2,3

## 2. Most Required Skills Across Each Industry, Job Level, and Location (State)
## 


- Job Level, Industry, Company, Location, Skill, Work Mode

In [30]:
df_major = df.copy()

In [31]:
df_major = df_major.explode("job_skills_list")

df_major["skill"] = (
    df_major["job_skills_list"]
    .astype(str)
    .str.strip()
    .str.lower()
)

skill_key = df_major["skill"].str.replace(r"\s+", "", regex=True)
df_major.loc[skill_key == "problemsolving", "skill"] = "problem solving"

df_major["company_major_industry"] = df_major["company_major_industry"].fillna("Other")
df_major["job_level"] = df_major["job_level"].fillna("Unknown")
df_major["state"] = df_major["state"].fillna("Unknown")
df_major["job_type"] = df_major["job_type"].fillna("Unknown")

aggregatedByIndustryAndLevel_major = (
    df_major
    .groupby(
        ["job_level", "company_major_industry", "company", "state", "skill", "job_type"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "count"})
)
aggregatedByIndustryAndLevel_major = aggregatedByIndustryAndLevel_major.sort_values(
    "count", ascending=False
)

In [27]:
aggregatedByIndustryAndLevel_major

,job_level,company_major_industry,company,state,skill,job_type,count
4484574,Mid senior,"Mining, Quarrying, and Oil and Gas Extraction",Dollar General,TX,customer service,Onsite,1118
4485843,Mid senior,"Mining, Quarrying, and Oil and Gas Extraction",Dollar General,TX,supervisory experience,Onsite,927
4456076,Mid senior,"Mining, Quarrying, and Oil and Gas Extraction",Dollar General,FL,customer service,Onsite,835
4485055,Mid senior,"Mining, Quarrying, and Oil and Gas Extraction",Dollar General,TX,mathematical calculations,Onsite,814
1738403,Mid senior,Health Care and Social Assistance,Health eCareers,CA,nursing,Onsite,801
...,...,...,...,...,...,...,...
2320052,Mid senior,Information,American National,TX,treasury,Onsite,1
2320051,Mid senior,Information,American National,TX,token/session exploitation,Onsite,1
2320050,Mid senior,Information,American National,TX,time management,Onsite,1
2320049,Mid senior,Information,American National,TX,testing,Onsite,1


In [32]:
# Partition aggregatedByIndustry into up to 10 CSVs, each under 50MB
import math

NUM_PARTITIONS = 10
MAX_SIZE_MB = 90
OUTPUT_DIR = "."
BASE_NAME = "aggregatedbyindustry"

df = aggregatedByIndustryAndLevel_major
n = len(df)
# Estimate CSV size from a sample to decide how many parts we need
sample_size = min(10_000, n)
sample_csv = df.head(sample_size).to_csv(index=False)
bytes_per_row = len(sample_csv.encode("utf-8")) / sample_size
total_bytes = bytes_per_row * n
min_parts_for_size = math.ceil(total_bytes / (MAX_SIZE_MB * 1024 * 1024))
num_parts = max(NUM_PARTITIONS, min_parts_for_size)

chunk_size = math.ceil(n / num_parts)
for i in range(num_parts):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, n)
    chunk = df.iloc[start:end]
    path = os.path.join(OUTPUT_DIR, f"{BASE_NAME}_part_{i + 1:02d}.csv")
    chunk.to_csv(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{path}: {chunk.shape[0]} rows, {size_mb:.2f} MB")
print(f"Done: {num_parts} CSV(s) written (each under {MAX_SIZE_MB} MB).")

./aggregatedbyindustry_part_01.csv: 656874 rows, 47.68 MB
./aggregatedbyindustry_part_02.csv: 656874 rows, 47.08 MB
./aggregatedbyindustry_part_03.csv: 656874 rows, 47.72 MB
./aggregatedbyindustry_part_04.csv: 656874 rows, 46.94 MB
./aggregatedbyindustry_part_05.csv: 656874 rows, 45.47 MB
./aggregatedbyindustry_part_06.csv: 656874 rows, 52.29 MB
./aggregatedbyindustry_part_07.csv: 656874 rows, 55.05 MB
./aggregatedbyindustry_part_08.csv: 656874 rows, 47.85 MB
./aggregatedbyindustry_part_09.csv: 656874 rows, 47.30 MB
./aggregatedbyindustry_part_10.csv: 656870 rows, 46.22 MB
Done: 10 CSV(s) written (each under 90 MB).


In [17]:
aggregatedByIndustryAndLevel_major.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11664380 entries, 4992461 to 630202
Data columns (total 8 columns):
 #   Column                  Dtype 
---  ------                  ----- 
 0   job_level               object
 1   company_major_industry  object
 2   company                 object
 3   state                   object
 4   skill                   object
 5   job_type                object
 6   count                   int64 
 7   skills_number           int64 
dtypes: int64(2), object(6)
memory usage: 800.9+ MB
